# Phân loại trên dữ liệu đã trích xuất đặc trưng — grid search 11 mô hình

Chạy grid search tất cả mô hình phân loại trên 5 fold của **mọi phương pháp trích xuất đặc trưng × mọi bộ dữ liệu**
trong một thư mục số chiều (vd `fe_pca_gse68465`, `fe_chnmf_tcga_luad`, ...). Cùng mô hình, lưới tham số, quy trình
và độ đo với `clf_gridsearch_fs_group.ipynb` để so sánh trực tiếp với các phương pháp chọn gene.

- **Đầu vào — `FE_DIR`** (chỉ cần sửa dòng này): thư mục số chiều của `reduced-dimension-data`,
  vd `/kaggle/input/datasets/nguynthlun/extracted-features/100`. Bên trong mỗi phần tử là **thư mục**
  `fe_<method>_<bo>/fold_k/{train,val,test}.h5` (do skill `organizing-fe-results` tạo) hoặc **file zip** `fe_<method>_<bo>.zip`
  gốc từ notebook trích xuất (có `features/fold_k/...`). Notebook tự tìm `fold_k/train.h5` ở mọi độ sâu
  (Kaggle đôi khi lồng thêm 1 cấp thư mục).
  Tên bộ dữ liệu lấy từ hậu tố tên: `fe_pca_gse68465` → `gse68465`; số chiều (`100`) lấy từ tên `FE_DIR`.
- **Khác bản FS:** không cần `DATA_ROOT` và danh sách gene — file h5 đã là dữ liệu sau giảm chiều
  (cột `PC1..`, `AE1..`, `CHNMF1..` + `label`), dùng **toàn bộ cột** trừ `label`. Đặc trưng đã được fit chỉ trên train
  ở notebook trích xuất; ở đây **không chuẩn hoá** thêm.
- **Mô hình** (lưới tham số tham khảo `model_tuning_5fold_kaggle.ipynb`): KNN, Gaussian NB, Logistic Regression,
  Decision Tree, Random Forest, Linear SVM, RBF SVM, MLP, **XGBoost**, **LightGBM**, và **Stacking**
  (Mahmoud & Takaoka, *Healthcare Analytics* 7 (2025) 100373: base learner MLP, RF, KNN, SVM → meta-learner XGBoost,
  meta học trên dự đoán out-of-fold của base learner).
- **Quy trình mỗi fold**: huấn luyện mọi cấu hình trên `train` → **chọn cấu hình theo macro-F1 trên `val`** →
  báo cáo trên `test` (test không dùng để chọn).
- **Độ đo** (val và test): Accuracy, Balanced Accuracy, Macro/Weighted Precision, Macro/Weighted Recall,
  Macro/Weighted F1, MCC, Cohen's Kappa, ROC-AUC (one-vs-rest, macro), Confusion Matrix (test),
  thời gian huấn luyện, thời gian suy luận.
- **Đầu ra** (`/kaggle/working/classification/<N>/extraction/`) — **cùng cột với bản FS** (`fs_group = "extraction"`,
  `fs_method` = phương pháp trích xuất, `n_genes` = số đặc trưng) để ghép bảng so sánh:
  - `results_extraction.csv` — mỗi dòng = (phương pháp, bộ, fold, mô hình) với cấu hình tốt nhất.
    Ghi dần sau mỗi mô hình: chạy lại notebook sẽ bỏ qua phần đã xong.
  - `summary_extraction.csv` — trung bình (± std) qua 5 fold cho từng (phương pháp, bộ, mô hình),
    kèm confusion matrix cộng dồn trên test của 5 fold.

In [ ]:
import sys, subprocess, importlib.util
# tables: backend cho pd.read_hdf | xgboost, lightgbm: Kaggle thuong co san
_missing = [p for p, m in [("tables", "tables"), ("xgboost", "xgboost"), ("lightgbm", "lightgbm")]
            if importlib.util.find_spec(m) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

## 1. Cấu hình

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Dau vao: thu muc so chieu chua cac fe_<method>_<bo>/fold_k/{train,val,test}.h5 (hoac zip fe_<method>_<bo>.zip)
FE_DIR = "/kaggle/input/datasets/nguynthlun/extracted-features/100"
OUT_ROOT = "/kaggle/working/classification"

KNOWN_DATASETS = ["gse68465", "tcga_luad", "tcga_lusc"]   # hau to ten de tach ten bo
N_FOLDS = 5
SEED = 42
N_JOBS = -1
SELECT_METRIC = "f1_macro"    # do do tren val de chon cau hinh
QUICK = False                 # True: luoi nho de chay thu nhanh
MODELS_TO_RUN = ["knn", "gaussian_nb", "logistic_regression", "decision_tree", "random_forest",
                 "linear_svm", "rbf_svm", "mlp", "xgboost", "lightgbm", "stacking"]
FS_RUNS_TO_RUN = None         # None = tat ca; hoac vd ["fe_pca_gse68465"]

FS_GROUP = "extraction"
N_FEATURES_TAG = os.path.basename(os.path.normpath(FE_DIR))   # vd "100"
OUT_DIR = os.path.join(OUT_ROOT, N_FEATURES_TAG, FS_GROUP)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, f"results_{FS_GROUP}.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, f"summary_{FS_GROUP}.csv")

assert os.path.isdir(FE_DIR), f"Khong thay FE_DIR={FE_DIR} -- da Add Input chua?"
print("Trich xuat:", FE_DIR, "| so chieu:", N_FEATURES_TAG)
print("Ket qua   :", RESULTS_CSV)
print("So CPU    :", os.cpu_count())

## 2. Tìm các phương pháp trích xuất trong `FE_DIR` (giải nén zip nếu cần)

In [ ]:
import glob, json, time, zipfile, itertools, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}
SPLITS = ["train", "val", "test"]
EXTRACT_DIR = os.path.join("/kaggle/working/_fe_extracted", N_FEATURES_TAG)


def parse_run_name(entry):
    # "fe_pca_gse68465(.zip)" -> ("fe_pca_gse68465", "pca", "gse68465")
    stem = entry[:-4] if entry.endswith(".zip") else entry
    for ds in sorted(KNOWN_DATASETS, key=len, reverse=True):
        if stem.endswith("_" + ds):
            method = stem[:-len(ds) - 1]
            return stem, (method[3:] if method.startswith("fe_") else method), ds
    return None


def find_fold_dirs(root):
    # fold_k/{train,val,test}.h5 o bat ky do sau nao (features/fold_k trong zip, hoac fold_k truc tiep)
    out = {}
    for r in range(1, N_FOLDS + 1):
        hits = sorted(glob.glob(os.path.join(root, "**", f"fold_{r}", "train.h5"), recursive=True))
        assert hits, f"Khong thay fold_{r}/train.h5 trong {root}"
        d = os.path.dirname(hits[0])
        for s in SPLITS:
            assert os.path.isfile(os.path.join(d, f"{s}.h5")), f"Khong thay {d}/{s}.h5"
        out[r] = d
    return out


FS_RUNS = {}
for entry in sorted(os.listdir(FE_DIR)):   # thu muc "x" dung truoc "x.zip" -> uu tien ban da giai nen
    path = os.path.join(FE_DIR, entry)
    parsed = parse_run_name(entry)
    if parsed is None or parsed[0] in FS_RUNS:
        continue
    stem, method, ds = parsed
    if FS_RUNS_TO_RUN is not None and stem not in FS_RUNS_TO_RUN:
        continue
    if entry.endswith(".zip"):
        root = os.path.join(EXTRACT_DIR, stem)
        if not os.path.isdir(root):
            # chi giai nen features/ (du lieu sau giam chieu) va config.json
            with zipfile.ZipFile(path) as zf:
                zf.extractall(root, members=[m for m in zf.namelist()
                                             if m.startswith("features/") or m.endswith("config.json")])
    elif os.path.isdir(path):
        root = path
    else:
        continue
    FS_RUNS[stem] = {"method": method, "dataset": ds, "folds": find_fold_dirs(root)}

assert FS_RUNS, f"Khong tim thay phuong phap trich xuat nao trong {FE_DIR}"
print(pd.DataFrame([{"fe_run": k, "method": v["method"], "dataset": v["dataset"]}
                    for k, v in FS_RUNS.items()]).to_string(index=False))

## 3. Đọc dữ liệu

In [ ]:
_DATA_CACHE = {}


def load_run(stem):
    # Chi giu du lieu cua 1 fe_run trong RAM (moi phuong phap x bo co file h5 rieng)
    if stem not in _DATA_CACHE:
        _DATA_CACHE.clear()
        _DATA_CACHE[stem] = {r: {s: pd.read_hdf(os.path.join(d, f"{s}.h5"), key="data") for s in SPLITS}
                             for r, d in FS_RUNS[stem]["folds"].items()}
    return _DATA_CACHE[stem]


def get_xy(df, feats):
    missing = [f for f in feats if f not in df.columns]
    assert not missing, f"{len(missing)} dac trung khong co trong du lieu, vd {missing[:5]}"
    # Dac trung da trich xuat (fit tren train o notebook trich xuat), KHONG chuan hoa them
    return df[feats].values.astype(np.float64), df["label"].values.astype(int)


# Kiem tra nhanh: so dac trung cua tung fe_run (fold 1)
for stem, info in FS_RUNS.items():
    cols = pd.read_hdf(os.path.join(info["folds"][1], "train.h5"), key="data").columns
    info["n_features"] = len(cols) - ("label" in cols)
    print(f"{stem:30s} {info['n_features']} dac trung | vd {list(cols[:3])}")

## 4. Mô hình và lưới tham số

Lưới lấy từ `model_tuning_5fold_kaggle.ipynb`, chỉnh lại khoảng `C`/`gamma` cho phù hợp dữ liệu ~100 chiều
(lưới cũ dò cho ~20.000 gene: `gamma="scale"` giờ cỡ 1e-3 thay vì 1e-5). Cấu hình mặc định luôn được chạy kèm.
XGBoost không có `class_weight` nên `class_weight="balanced"` được thực hiện bằng `sample_weight`.

**Stacking** (theo bài báo): base learner = MLP, RF, KNN, SVM — dùng **cấu hình tốt nhất trên val của
cùng fold** (SVM = linear hoặc RBF, cái nào val macro-F1 cao hơn); meta-learner = XGBoost, học trên dự đoán
out-of-fold (StratifiedKFold trên train) của base learner; chỉ dò lưới tham số của meta-learner.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SVM_MAX_ITER = 1_000_000   # tranh libsvm chay vo han tren du lieu khong chuan hoa


def build_model(model, params):
    p = dict(params)
    if model == "knn":
        return KNeighborsClassifier(n_jobs=N_JOBS, **p)
    if model == "gaussian_nb":
        return GaussianNB(**p)
    if model == "logistic_regression":
        return LogisticRegression(max_iter=5000, random_state=SEED, **p)
    if model == "decision_tree":
        return DecisionTreeClassifier(random_state=SEED, **p)
    if model == "random_forest":
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if model == "linear_svm":
        return SVC(kernel="linear", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "rbf_svm":
        return SVC(kernel="rbf", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "mlp":
        p["hidden_layer_sizes"] = tuple(p["hidden_layer_sizes"])
        return MLPClassifier(max_iter=500, random_state=SEED, **p)
    if model == "xgboost":
        p.pop("class_weight", None)   # xu ly bang sample_weight trong fit_model
        return XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, verbosity=0, **p)
    if model == "lightgbm":
        return LGBMClassifier(n_jobs=N_JOBS, random_state=SEED, verbose=-1, **p)
    raise ValueError(model)


def fit_model(model, params, X, y):
    est = build_model(model, params)
    kw = {}
    if model == "xgboost" and params.get("class_weight") == "balanced":
        kw["sample_weight"] = compute_sample_weight("balanced", y)
    t0 = time.perf_counter()
    est.fit(X, y, **kw)
    return est, time.perf_counter() - t0


DEFAULTS = {
    "knn": {"n_neighbors": 5, "weights": "uniform", "metric": "minkowski"},
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "class_weight": None},
    "decision_tree": {"max_depth": None, "min_samples_leaf": 1, "criterion": "gini", "class_weight": None},
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 1,
                      "class_weight": None, "max_samples": None},
    "linear_svm": {"C": 1.0, "class_weight": None},
    "rbf_svm": {"C": 1.0, "gamma": "scale", "class_weight": None},
    "mlp": {"hidden_layer_sizes": [100], "alpha": 1e-4},
    "xgboost": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3, "subsample": 1.0,
                "colsample_bytree": 1.0, "class_weight": None},
    "lightgbm": {"n_estimators": 100, "num_leaves": 31, "learning_rate": 0.1, "min_child_samples": 20,
                 "class_weight": None},
    # stacking: tham so cua meta-learner XGBoost
    "stacking": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3},
}

if QUICK:
    GRIDS = {
        "knn": {"n_neighbors": [5, 11], "weights": ["distance"], "metric": ["minkowski", "cosine"]},
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-3]},
        "logistic_regression": {"C": [0.01, 1.0], "class_weight": ["balanced"]},
        "decision_tree": {"max_depth": [None, 5], "min_samples_leaf": [5], "criterion": ["gini"],
                          "class_weight": ["balanced"]},
        "random_forest": {"n_estimators": [300], "max_features": ["sqrt"], "min_samples_leaf": [1, 5],
                          "class_weight": ["balanced"], "max_samples": [None]},
        "linear_svm": {"C": [1e-3, 1e-1], "class_weight": ["balanced"]},
        "rbf_svm": {"C": [1.0, 10.0], "gamma": ["scale"], "class_weight": ["balanced"]},
        "mlp": {"hidden_layer_sizes": [[100]], "alpha": [1e-4, 1.0]},
        "xgboost": {"n_estimators": [200], "max_depth": [3], "learning_rate": [0.1], "subsample": [0.8],
                    "colsample_bytree": [0.5], "class_weight": [None, "balanced"]},
        "lightgbm": {"n_estimators": [200], "num_leaves": [15], "learning_rate": [0.1],
                     "min_child_samples": [10], "class_weight": [None, "balanced"]},
        "stacking": {"n_estimators": [100], "max_depth": [3], "learning_rate": [0.1]},
    }
else:
    GRIDS = {
        "knn": {"n_neighbors": [3, 5, 7, 11, 15, 21], "weights": ["uniform", "distance"],
                "metric": ["minkowski", "cosine"]},                                    # 24
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]},         # 6
        "logistic_regression": {"C": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0],
                                "class_weight": [None, "balanced"]},                   # 12
        "decision_tree": {"max_depth": [None, 3, 5, 10, 20], "min_samples_leaf": [1, 5, 10],
                          "criterion": ["gini", "entropy"],
                          "class_weight": [None, "balanced"]},                         # 60
        "random_forest": {"n_estimators": [500], "max_features": ["sqrt", "log2", 0.3],
                          "min_samples_leaf": [1, 5, 10], "class_weight": ["balanced"],
                          "max_samples": [None, 0.5]},                                 # 18
        "linear_svm": {"C": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                       "class_weight": [None, "balanced"]},                            # 12
        "rbf_svm": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", 1e-4, 1e-3, 1e-2],
                    "class_weight": [None, "balanced"]},                               # 32
        "mlp": {"hidden_layer_sizes": [[64], [256], [256, 64]],
                "alpha": [1e-4, 1e-2, 1.0, 10.0]},                                     # 12
        "xgboost": {"n_estimators": [200, 500], "max_depth": [3, 6], "learning_rate": [0.05, 0.1],
                    "subsample": [0.8], "colsample_bytree": [0.5, 1.0],
                    "class_weight": [None, "balanced"]},                               # 32
        "lightgbm": {"n_estimators": [200, 500], "num_leaves": [7, 15], "learning_rate": [0.05, 0.1],
                     "min_child_samples": [5, 20], "class_weight": [None, "balanced"]}, # 32
        "stacking": {"n_estimators": [100, 300], "max_depth": [2, 3],
                     "learning_rate": [0.05, 0.1]},                                    # 8
    }


def configs_of(model):
    grid = GRIDS[model]
    cfgs = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]
    return cfgs if DEFAULTS[model] in cfgs else [DEFAULTS[model]] + cfgs


CONFIGS = {m: configs_of(m) for m in MODELS_TO_RUN}
n_cfg = sum(len(c) for c in CONFIGS.values())
print(pd.Series({m: len(c) for m, c in CONFIGS.items()}).to_string())
print(f"Tong: {n_cfg} cau hinh x {N_FOLDS} fold x {len(FS_RUNS)} (phuong phap trich xuat x bo) "
      f"= {n_cfg * N_FOLDS * len(FS_RUNS)} lan huan luyen (chua tinh CV noi bo cua stacking)")

### Stacking: base MLP, RF, KNN, SVM → meta XGBoost

In [ ]:
STACK_BASES = ["mlp", "random_forest", "knn", "svm"]   # "svm" = linear_svm / rbf_svm tot hon tren val
STACK_CV = 5


def stack_method(model):
    return "decision_function" if model in ("linear_svm", "rbf_svm") else "predict_proba"


def base_output(est, model, X):
    return getattr(est, stack_method(model))(X).reshape(len(X), -1)


class StackedModel:
    # base learner da fit tren toan bo train + meta XGBoost fit tren du doan out-of-fold
    def __init__(self, bases, meta):
        self.bases, self.meta = bases, meta

    def _meta_features(self, X):
        return np.hstack([base_output(est, m, X) for m, est in self.bases])

    def predict(self, X):
        return self.meta.predict(self._meta_features(X))

    def predict_proba(self, X):
        return self.meta.predict_proba(self._meta_features(X))


def fit_stack_bases(bases, X, y):
    # bases: [(model, params)] -> (base da fit, ma tran out-of-fold cho meta, thoi gian)
    n_splits = max(2, min(STACK_CV, np.bincount(y).min()))   # lop hiem (vd LUSC stage IV ~5 mau)
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=SEED)
    t0 = time.perf_counter()
    oof, fitted = [], []
    for model, params in bases:
        oof.append(cross_val_predict(build_model(model, params), X, y, cv=cv,
                                     method=stack_method(model)).reshape(len(y), -1))
        fitted.append((model, build_model(model, params).fit(X, y)))
    return fitted, np.hstack(oof), time.perf_counter() - t0


def fit_stacking(stack_state, meta_params, y):
    fitted, oof, base_s = stack_state
    meta, meta_s = fit_model("xgboost", meta_params, oof, y)
    return StackedModel(fitted, meta), base_s + meta_s

## 5. Độ đo

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
                             confusion_matrix)

METRIC_NAMES = ["accuracy", "balanced_accuracy", "precision_macro", "precision_weighted",
                "recall_macro", "recall_weighted", "f1_macro", "f1_weighted", "mcc",
                "cohen_kappa", "roc_auc"]


def get_scores(est, X):
    # xac suat (hoac decision_function voi SVC) de tinh ROC-AUC; cot j <-> lop ma hoa j
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X)
    d = est.decision_function(X)
    return np.column_stack([-d, d]) if d.ndim == 1 else d


def roc_auc_ovr_macro(y, scores):
    # one-vs-rest, trung binh macro; bo qua lop khong co mat trong y (AUC khong xac dinh)
    aucs = [roc_auc_score(y == c, scores[:, c]) for c in range(scores.shape[1])
            if 0 < (y == c).sum() < len(y)]
    return float(np.mean(aucs)) if aucs else np.nan


def compute_metrics(y, p, scores):
    kw = dict(zero_division=0)
    return {
        "accuracy": accuracy_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "precision_macro": precision_score(y, p, average="macro", **kw),
        "precision_weighted": precision_score(y, p, average="weighted", **kw),
        "recall_macro": recall_score(y, p, average="macro", **kw),
        "recall_weighted": recall_score(y, p, average="weighted", **kw),
        "f1_macro": f1_score(y, p, average="macro", **kw),
        "f1_weighted": f1_score(y, p, average="weighted", **kw),
        "mcc": matthews_corrcoef(y, p),
        "cohen_kappa": cohen_kappa_score(y, p),
        "roc_auc": roc_auc_ovr_macro(y, scores),
    }


# Do do chon cau hinh tren val (chi dung nhan du doan)
SCORERS = {
    "f1_macro": lambda y, p: f1_score(y, p, average="macro", zero_division=0),
    "balanced_accuracy": balanced_accuracy_score,
    "mcc": matthews_corrcoef,
    "accuracy": accuracy_score,
}
select_score = SCORERS[SELECT_METRIC]


def evaluate(est, X, y):
    t0 = time.perf_counter()
    p = est.predict(X)
    infer_s = time.perf_counter() - t0
    return compute_metrics(y, p, get_scores(est, X)), infer_s, p

## 6. Grid search trên 5 fold

Mỗi (phương pháp trích xuất × bộ, fold, mô hình): thử mọi cấu hình, fit trên `train`, chọn theo `SELECT_METRIC` trên `val`,
rồi đánh giá cấu hình tốt nhất trên `val` và `test`. `train_time_s` = thời gian fit cấu hình tốt nhất
(stacking: gồm CV out-of-fold + fit base + fit meta), `infer_time_test_s` = thời gian `predict` trên test,
`tuning_time_s` = tổng thời gian dò lưới của mô hình đó. Mỗi mô hình xong được ghi ngay vào `RESULTS_CSV`.

In [ ]:
def grid_search(fit_fn, configs, X_va, y_va):
    best, t0 = None, time.perf_counter()
    for params in configs:
        est, fit_s = fit_fn(params)
        score = select_score(y_va, est.predict(X_va))
        if best is None or score > best["score"]:
            best = {"params": params, "est": est, "fit_s": fit_s, "score": score}
    best["tuning_s"] = time.perf_counter() - t0
    return best


def make_row(stem, info, r, model, best, n_configs, sets, classes, feats):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = sets
    m_va, _, _ = evaluate(best["est"], X_va, y_va)
    m_te, infer_s, p_te = evaluate(best["est"], X_te, y_te)
    cm = confusion_matrix(y_te, p_te, labels=np.arange(len(classes)))
    return {
        "n_features": N_FEATURES_TAG, "fs_group": FS_GROUP, "fs_method": info["method"], "fs_run": stem,
        "dataset": info["dataset"], "fold": r, "model": model, "n_genes": len(feats),   # = so dac trung trich xuat (giu ten cot cua ban FS)
        "n_train": len(y_tr), "n_val": len(y_va), "n_test": len(y_te),
        "best_params": json.dumps(best["params"]), "n_configs": n_configs,
        "train_time_s": best["fit_s"], "tuning_time_s": best["tuning_s"],
        "infer_time_test_s": infer_s, "infer_time_per_sample_ms": 1e3 * infer_s / len(y_te),
        **{f"val_{k}": v for k, v in m_va.items()},
        **{f"test_{k}": v for k, v in m_te.items()},
        "class_labels": json.dumps([STAGE_NAMES.get(c, str(c)) for c in classes]),
        "confusion_matrix_test": json.dumps(cm.tolist()),
    }


def append_row(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", index=False, header=not os.path.exists(RESULTS_CSV))

In [ ]:
prev = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(
    columns=["fs_run", "fold", "model", "best_params", f"val_{SELECT_METRIC}"])
done = set(zip(prev["fs_run"], prev["fold"].astype(int), prev["model"]))
if done:
    print(f"Tiep tuc: da co {len(done)} dong trong {RESULTS_CSV}")

t_start = time.time()
for stem, info in sorted(FS_RUNS.items(), key=lambda kv: (kv[1]["dataset"], kv[0])):
    for r in range(1, N_FOLDS + 1):
        todo = [m for m in MODELS_TO_RUN if (stem, r, m) not in done]
        if not todo:
            continue
        fold = load_run(stem)[r]
        feats = [c for c in fold["train"].columns if c != "label"]   # toan bo dac trung trich xuat
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = [get_xy(fold[s], feats) for s in SPLITS]

        # Ma hoa nhan ve 0..K-1 theo cac lop cua train (XGBoost yeu cau; GSE68465 khong co stage IV)
        classes = np.unique(y_tr)
        assert set(y_va) <= set(classes) and set(y_te) <= set(classes), f"{stem} fold {r}: val/test co lop la"
        y_tr, y_va, y_te = [np.searchsorted(classes, y) for y in (y_tr, y_va, y_te)]
        sets = [(X_tr, y_tr), (X_va, y_va), (X_te, y_te)]

        # cau hinh tot nhat cua fold nay (tu lan chay truoc neu co) -> base learner cho stacking
        fold_best = {row.model: (json.loads(row.best_params), getattr(row, f"val_{SELECT_METRIC}"))
                     for row in prev[(prev.fs_run == stem) & (prev.fold == r)].itertuples()}

        for model in [m for m in todo if m != "stacking"] + [m for m in todo if m == "stacking"]:
            t0 = time.time()
            if model == "stacking":
                svm = max(["linear_svm", "rbf_svm"], key=lambda m: fold_best.get(m, (None, -1))[1])
                bases = []
                for b in STACK_BASES:
                    b = svm if b == "svm" else b
                    if b not in fold_best:
                        print(f"  [stacking] chua co cau hinh tot nhat cua {b}, dung DEFAULTS")
                    bases.append((b, fold_best.get(b, (DEFAULTS[b], None))[0]))
                state = fit_stack_bases(bases, X_tr, y_tr)
                best = grid_search(lambda p: fit_stacking(state, p, y_tr), CONFIGS[model], X_va, y_va)
                best["params"] = {"meta_xgboost": best["params"], "bases": {b: p for b, p in bases}}
            else:
                best = grid_search(lambda p: fit_model(model, p, X_tr, y_tr), CONFIGS[model], X_va, y_va)
            row = make_row(stem, info, r, model, best, len(CONFIGS[model]), sets, classes, feats)
            append_row(row)
            fold_best[model] = (best["params"], row[f"val_{SELECT_METRIC}"])
            done.add((stem, r, model))
            print(f"{stem} | fold {r} | {model:20s} val F1 {row['val_f1_macro']:.3f} | "
                  f"test F1 {row['test_f1_macro']:.3f} acc {row['test_accuracy']:.3f} "
                  f"AUC {row['test_roc_auc']:.3f} | {time.time() - t0:.0f}s "
                  f"(tong {(time.time() - t_start) / 60:.1f} phut)", flush=True)

print("Xong grid search:", RESULTS_CSV)

## 7. Tổng hợp: trung bình qua 5 fold

`summary_<nhom>.csv`: mỗi dòng = (phương pháp trích xuất, bộ, mô hình); cột độ đo = trung bình 5 fold,
cột `*_std` = độ lệch chuẩn của độ đo test, `confusion_matrix_test_sum` = cộng confusion matrix test
của 5 fold (mỗi mẫu làm test đúng 1 lần).

In [ ]:
res = pd.read_csv(RESULTS_CSV)
KEYS = ["n_features", "fs_group", "fs_method", "dataset", "model"]
TIME_COLS = ["train_time_s", "tuning_time_s", "infer_time_test_s", "infer_time_per_sample_ms"]
VAL_COLS = [f"val_{m}" for m in METRIC_NAMES]
TEST_COLS = [f"test_{m}" for m in METRIC_NAMES]


def sum_cm(s):
    mats = [np.array(json.loads(x)) for x in s]
    return json.dumps(np.sum(mats, axis=0).tolist()) if len({m.shape for m in mats}) == 1 else ""


g = res.groupby(KEYS, sort=False)
summary = g[TEST_COLS].mean()
for c in TEST_COLS:
    summary[f"{c}_std"] = g[c].std()
summary = summary[[c for t in TEST_COLS for c in (t, f"{t}_std")]]
summary = summary.join(g[VAL_COLS + TIME_COLS].mean())
summary["n_folds"] = g.size()
summary["class_labels"] = g["class_labels"].first()
summary["confusion_matrix_test_sum"] = g["confusion_matrix_test"].apply(sum_cm)
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
print("Da luu:", SUMMARY_CSV, summary.shape)

# Bang nhanh: test macro-F1 trung binh, hang = (phuong phap trich xuat, bo), cot = mo hinh
display(summary.pivot_table(index=["fs_method", "dataset"], columns="model", values="test_f1_macro")
        .reindex(columns=[m for m in MODELS_TO_RUN if m in set(summary.model)]).round(3))

In [ ]:
import shutil
zip_out = shutil.make_archive(os.path.join("/kaggle/working", f"classification_{N_FEATURES_TAG}_{FS_GROUP}"),
                              "zip", OUT_DIR)
print("Da nen:", zip_out, f"({os.path.getsize(zip_out) / 1e6:.2f} MB)")